In [ ]:
# === CELL 1: Install Libraries (Fixed for Python 3.12) ===
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jxq0sdtz/unsloth_fe19eb27b2704aaaacc712e18b659292
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jxq0sdtz/unsloth_fe19eb27b2704aaaacc712e18b659292
  Resolved https://github.com/unslothai/unsloth.git to commit 8ea81297cb4d93a35cd1f32ca1f97c67544ff76e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 147.4 MB/s eta 0:00:00

In [ ]:
# === CELL 2: Setup & Train ===
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load Base Model (Llama 3.2 3B - Optimized)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Add Adapters (Efficient Training)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",
)

# 3. Load Data (Counsel Chat)
dataset = load_dataset("nbertagnolli/counsel-chat", split="train")

# 4. Format Data (The "Therapist" Style)
def format_prompts(examples):
    instructions = "You are an empathetic mental health assistant."
    inputs = examples["questionText"]
    outputs = examples["answerText"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Using Llama 3 standard prompt format
        text = f"<|start_header_id|>user<|end_header_id|>\n\n{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

# 5. Train
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = dataset, dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 60,
        learning_rate = 2e-4, fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), logging_steps = 1,
        output_dir = "outputs",

        # --- THE FIX ---
        report_to = "none",  # <--- This disables the WandB login prompt
    ),
)
trainer.train()

# === CELL 3: Save & Convert to GGUF ===
print("💾 Saving Model to GGUF format...")
# This converts the model to a format your RTX 3050 can run efficiently
model.save_pretrained_gguf("model_final", tokenizer, quantization_method = "q4_k_m")

In [ ]:
# === RESCUE SCRIPT: Convert Saved Model to GGUF ===
from unsloth import FastLanguageModel
import torch

# 1. Reload the model from your saved folder (NOT retraining)
print("♻️ Reloading your trained model from 'model_final'...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "model_final", # Loading from LOCAL folder
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Force conversion with lower RAM usage
print("💾 Attempting conversion to GGUF again...")
model.save_pretrained_gguf(
    "model_final_rescued", # Saving to a NEW folder to avoid conflicts
    tokenizer,
    quantization_method = "q4_k_m"
)

print("✅ SUCCESS! Check the 'model_final_rescued' folder for your .gguf file.")

In [ ]:
import os
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",
)

dataset = load_dataset("nbertagnolli/counsel-chat", split="train")
def format_prompts(examples):
    inputs = examples["questionText"]
    outputs = examples["answerText"]
    texts = []
    for input, output in zip(inputs, outputs):
        text = f"<|start_header_id|>user<|end_header_id|>\n\n{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }
dataset = dataset.map(format_prompts, batched = True)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = dataset, dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 60, learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), logging_steps = 1,
        output_dir = "outputs", report_to = "none",
    ),
)
trainer.train()

drive_path = "/content/drive/MyDrive/MentalHealthBot_Adapters"
print(f" Saving adapters to {drive_path}...")
model.save_pretrained(drive_path)
tokenizer.save_pretrained(drive_path)
print(" Saved! Now go to Phase 2.")

==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.12.6 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.
Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/2775 [00:00<?, ? examples/s]

Map:   0%|          | 0/2775 [00:00<?, ? examples/s]

Map:   0%|          | 0/2775 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,775 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.003500
2,2.937300
3,2.927800
4,2.960900
5,3.003800
6,2.630000
7,2.709000
8,3.005600
9,2.960300
10,2.908500


💾 Saving adapters to /content/drive/MyDrive/MentalHealthBot_Adapters...
✅ Saved! Now go to Phase 2.


In [ ]:
# === PHASE 3: Low-RAM Conversion Strategy ===
import os
import gc
import torch
from unsloth import FastLanguageModel

# 1. Setup paths
drive_path = "/content/drive/MyDrive/MentalHealthBot_Adapters"
temp_merged_path = "/content/temp_merged_model"

print("♻️ Loading adapters from Google Drive...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = drive_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Merge and Save ONLY (No conversion yet)
print("💾 Merging and saving 16-bit model to disk...")
model.save_pretrained_merged(
    temp_merged_path,
    tokenizer,
    save_method = "merged_16bit", # Saves the raw fused model
)

# 3. CRITICAL STEP: Delete model to free RAM
print("🧹 Cleaning up RAM for conversion...")
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("✅ RAM freed! Starting conversion tool...")

# 4. Manual GGUF Conversion (Using shell commands to save memory)
# First, install llama.cpp directly
print("⚙️ Installing llama.cpp...")
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make clean && make all -j

# Convert the saved 16-bit model to GGUF format
print("🔨 Converting to GGUF (Quantization)...")
# Step A: Convert HF to GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py {temp_merged_path} --outfile {temp_merged_path}/model.gguf

# Step B: Quantize to q4_k_m (The format for your RTX 3050)
print("📉 Compressing to 4-bit...")
!./llama.cpp/llama-quantize {temp_merged_path}/model.gguf /content/model_final.gguf q4_k_m

print("🎉 DONE! You can now download 'model_final.gguf' from the main folder.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
♻️ Loading adapters from Google Drive...
==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.12.6 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


💾 Merging and saving 16-bit model to disk...


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:55<00:00, 27.53s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:38<00:00, 49.45s/it]


Unsloth: Merge process complete. Saved to `/content/temp_merged_model`
🧹 Cleaning up RAM for conversion...
✅ RAM freed! Starting conversion tool...
⚙️ Installing llama.cpp...
Cloning into 'llama.cpp'...
remote: Enumerating objects: 72595, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 72595 (delta 32), reused 26 (delta 26), pack-reused 72528 (from 4)
Receiving objects: 100% (72595/72595), 242.95 MiB | 23.51 MiB/s, done.
Resolving deltas: 100% (52504/52504), done.
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
🔨 Converting to GGUF (Quantization)...
INFO:hf-to-gguf:Loading model: temp_merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-

In [ ]:
# 1. Clean up
!rm -rf llama.cpp

# 2. Download repo
!git clone https://github.com/ggerganov/llama.cpp

# 3. Build using CMAKE (The new way)
# We create a build directory, configure it, and compile the 'llama-quantize' tool
!cd llama.cpp && cmake -B build && cmake --build build --config Release --target llama-quantize

# 4. Run the conversion
# Note: The tool is now located inside the 'build/bin' folder
print("📉 Compressing model...")
!./llama.cpp/build/bin/llama-quantize /content/temp_merged_model/model.gguf /content/model_final.gguf q4_k_m

print("🎉 DONE! Refresh file browser and download 'model_final.gguf'")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 72595, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 72595 (delta 32), reused 26 (delta 26), pack-reused 72528 (from 4)
Receiving objects: 100% (72595/72595), 242.95 MiB | 18.64 MiB/s, done.
Resolving deltas: 100% (52504/52504), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assemb